## KVCache manager

Build a simplified manager to store and evict Key-Value tensors 
for multi-turn conversations, 
implementing an LRU (Least Recently Used) policy to handle VRAM limits.


```python
class KVCacheManager:
    def __init__(self, max_blocks: int, block_size: int):
        """
        max_blocks: Total number of KV blocks the "VRAM" can hold.
        block_size: The dimension/size of each block (e.g., sequence length or hidden dim).
        """
        self.max_blocks = max_blocks
        self.block_size = block_size
        
        # This will be your primary storage
        # Suggestion: Use an OrderedDict to implement LRU efficiently in Python
        self.cache = OrderedDict() 
        
    def get_kv(self, request_id: str) -> Optional[Any]:
        """Retrieve KV tensors for a request and update its 'recency'."""
        pass

    def update_kv(self, request_id: str, kv_tensor: Any):
        """Add or update KV tensors. If full, evict the least recently used."""
        pass

    def evict(self):
        """The core LRU logic."""
        pass
```

In [ ]:
import time
import asyncio
from collections import OrderedDict
from typing import Any, Optional
from dataclasses import dataclass


@dataclass
class Request:
    request_id: str
    future: asyncio.Future[Any]
    arrival_time: float


class TieredKVCacheManager:
    def __init__(self, max_gpu_blocks:int, max_cpu_blocks:int, block_size:int):
        self.max_gpu_blocks = max_gpu_blocks
        self.max_cpu_blocks = max_cpu_blocks
        self.block_size = block_size
        
        self._gpu_cache = OrderedDict() 
        self._cpu_cache = OrderedDict() 

        # I will queue up the requests
        self._queue = asyncio.Queue(maxsize=max_gpu_blocks + max_cpu_blocks)
        self._task = None
    
    async def __aenter__(self) -> "TieredKVCacheManager":
        self._task = asyncio.create_task(self.get_kv_batch())
        return self
    
    async def __aexit__(self, exc_type, exc, tb) -> None:
        pass
    
    async def get_kv(self, request_id: str) -> Optional[Any]:
        loop = asyncio.get_running_loop()
        fut = loop.create_future()
        req = Request(
            request_id=request_id, 
            future=fut, 
            arrival_time=time.time()
        )

        await self._queue.put(req)

        return await fut

    async def _make_room_on_gpu(self):
        if len(self._gpu_cache) == self.max_gpu_blocks:
            k, v = self._gpu_cache.popitem(last=False)

            # make room on cpu
            if len(self._cpu_cache) == self.max_cpu_blocks:
                self._cpu_cache.popitem(last=False)
            
            self._cpu_cache[k] = v
            self._cpu_cache.move_to_end(k)

    async def get_kv_batch(self) -> None:
        while True:
            req = await self._queue.get()
            
            # req is in gpu cache
            if req.request_id in self._gpu_cache:
                self._gpu_cache.move_to_end(req.request_id)
                req.future.set_result(self._gpu_cache[req.request_id])
            # req is in cpu cache
            elif req.request_id in self._cpu_cache:
                await self._make_room_on_gpu()

                v = self._cpu_cache.pop(req.request_id)
                self._gpu_cache[req.request_id] = v
                self._gpu_cache.move_to_end(req.request_id)

                req.future.set_result(v)
            # req is not in cache
            else:
                res = await self._mock_model_inference(req.request_id)
                
                self._make_room_on_gpu()
                self._gpu_cache[req.request_id] = res
                self._gpu_cache.move_to_end(req.request_id)

                req.future.set_result(res)
